In [ ]:
from pathlib import Path

import pandas as pd

from experimental_hdf5_analysis import (
    analyze_experimental_manifest,
    collect_experimental_rate_isi_moments,
    export_experimental_summary_table,
    find_latest_experimental_manifest,
    summarize_experimental_batches,
)

In [ ]:
# ROOTS
EXPERIMENTAL_INPUT_ROOT = Path('EXP_DATA_HDF5_only_missing')  # Root containing the raw exported experimental trial HDF5 files. If sidecar analyses already exist elsewhere, this should still point to the raw-export root used by the manifest.
EXPERIMENTAL_ANALYZED_ROOT = Path('EXP_DATA_ANALYZED_HDF5')  # Root where the Python sidecar analysis HDF5 files and batch summaries are written.
MANIFEST_CSV_PATH = find_latest_experimental_manifest(EXPERIMENTAL_INPUT_ROOT)

# SUBSET FILTERS
PARTICIPANT_IDS =  None # e.g. ['P01'] # None for all participants  
SESSION_TAGS = None     # e.g. ['ses-S001']
CONDITION_IDS = None   # e.g. ['HAND_NOGO'] # None for all participants  
MAX_TRIALS = None       # e.g. 10 for a fast smoke test  # None for all trials 
MAX_BATCHES = None      # e.g. 1 for a fast smoke test; this limits participant x session x condition batch summaries/exports, not per-trial analysis.

# DISPLAY ONLY
DISPLAY_XLIM_REL_CUE_S = (-4.0, 1.0)  # Cue-relative x-limits for the batch summary figures only. None uses the full exported cue-relative window.

# FIXED OBSERVATION WINDOWS USED FOR PER-TRIAL FEATURE EXPORTS
OBSERVATION_WINDOW_CONFIG = {
    'normalization_win': (-4.0, -3.0),  # 1 s normalization-era observation window used for OBS_WINDOW_m4_to_m3_* features.
    'baseline_win': (-3.0, -2.0),  # 1 s baseline observation window used for OBS_WINDOW_m3_to_m2_* features.
    'baseline_extended_win': (-2.0, -1.0),  # 1 s extended-baseline observation window used for OBS_WINDOW_m2_to_m1_* features.
    'ready_win': (-1.0, 0.0),  # 1 s ready-period observation window used for OBS_WINDOW_m1_to_0_* features.
    'post_win': (0.0, 1.0),  # 1 s post-cue observation window used for OBS_WINDOW_0_to_1_* features.
}
OBS_BASELINE_WINDOW_REL_CUE_S = (-3.0, -1.0)  # Longer cue-relative window used for OBS_BASELINE_* CST summaries; separate from the CST modulation normalization window.

# ANALYSIS CONFIGURATION
ANALYSIS_CONFIG = {
    ## TIME WINDOWS FOR EACH ANALYSIS
    # Firing rates and ISI CV moments
    'firing_rate_isi_window_rel_cue_s': (-4.0, -1.0),  # Cue-relative window used to compute per-MU mean firing rates and ISI-CV values before the batch-level mean / SD / skew summaries are built.
    # Low-pass + Band-specifiic CST baseline & modulation
    'cst_modulation_normalization_window_rel_cue_s': (-4.0, -3.0), # -1.0),  # Cue-relative normalization window used for the alpha / beta Hilbert-envelope mean and SD, and therefore for CST modulation z-scores or percent-change traces.
    # Synchrony
    'sync_analysis_window_rel_cue_s': (-5.0, 2.0),  # Cue-relative support used to compute the sliding synchrony trace itself. Keep this wide enough to contain the sync baseline and post windows after accounting for half the synchrony window width.
    'sync_baseline_win_rel_cue_s': (-3.0, -1.0),  # Baseline window, relative to the cue, used for synchrony summary metrics and optional sync z-scoring.
    'sync_post_win_rel_cue_s': (0.0, 0.5),  # Post-cue window, relative to the cue, used to extract synchrony peak / integral summary metrics.
    'observation_window_defs_rel_cue_s': OBSERVATION_WINDOW_CONFIG,  # Named 1 s windows used to compute the saved OBS_WINDOW_* scalar features per trial.
    'obs_baseline_window_rel_cue_s': OBS_BASELINE_WINDOW_REL_CUE_S,  # Longer CST baseline-summary window used for OBS_BASELINE_* means and SDs in the export.
    # Feature extraction and spline-fitting windows: see FEATURE_EXTRACTION_CONFIG below.
    

    ## ANALYSIS-SPECIFIC PARAMETERS
    # Low-pass + Band-specifiic CST baseline & modulation
    'lf_band_hz': [0.0, 5.0],   # Low-pass / low-frequency band used for CST low-frequency analysis and summary plotting.
    'alpha_band_hz': [8.0, 13.0],  # Alpha band used for CST band-pass filtering and Hilbert-envelope modulation analysis.
    'beta_band_hz': [13.0, 30.0],  # Beta band used for CST band-pass filtering and Hilbert-envelope modulation analysis.
    'active_unit_rate_threshold_hz': 3.0,  # Shared active-unit threshold used to decide which MUs are kept for CST, synchrony, and ISI-CV. Mean firing-rate summaries remain computed across all MUs.
    'cst_modulation_mode': 'zscore',  # 'zscore' or 'percent'; controls how alpha/beta Hilbert envelopes are expressed relative to baseline in saved outputs and summary figures.
    'scale_filter_order_to_frequency': True,  # If True, CST filter order is adapted to frequency band; affects LF/alpha/beta CST filtering only.
    'filter_order_scaling_coeff': 1.0,  # Scaling coefficient used when scale_filter_order_to_frequency=True.
    'default_freq_filter_order': 6,  # Default Butterworth order used by CST band filters when no scaling override applies.
    'lowest_freq_filter_order': 2,  # Minimum allowed CST filter order after scaling/clamping.
    'max_freq_filter_order': 10,  # Maximum allowed CST filter order after scaling/clamping.
    # Synchrony
    'sync_trace_display_mode': 'absolute',  # 'absolute' uses the raw excess-coincidence index; 'zscore' baseline-normalizes the sync trace for display.
    'enable_sync_index_analysis': True,  # Master switch for the sliding synchrony / coincidence analysis.
    'sync_bin_ms': 1.0,  # Spike-train bin size for the synchrony analysis only.
    'sync_win_ms': 100.0,  # Sliding-window duration for the synchrony analysis only.
    'sync_step_ms': 10.0,  # Step size between synchrony windows; smaller values give finer temporal resolution but cost more runtime.
    'sync_coinc_lag_ms': 6.0,  # Coincidence tolerance used when counting near-synchronous spikes in the synchrony analysis.
    'sync_direction_mode': "symmetric_mean",   # 'legacy_forward' reproduces the historical asymmetric coincidence count; 'symmetric_mean' averages forward and reverse directions.
    'sync_expectation_mode': 'analytic',  # 'analytic' uses the fast Poisson-independence expectation; 'surrogate_circular_shift' is slower and recomputes chance coincidence from circular-shift surrogates.
    'sync_n_surrogates': 100,  # Number of surrogate windows used only when sync_expectation_mode='surrogate_circular_shift'.
    'sync_surrogate_min_shift_ms': 1.0,  # Minimum circular shift for surrogate synchrony estimation; only affects sync_expectation_mode='surrogate_circular_shift'.
    # MINIMAL OUTPUT OPTIONS
    'minimal_output': True,  # Compact-output profile for the derived sidecar analysis files only; the raw experimental export files stay untouched.
    'minimal_output_turn_off_figure_output': False,  # If True, no per-trial or per-batch figures are written even if the figure toggles below are True.
    'minimal_output_save_spike_trains': False,  # If False, the sidecar file does not duplicate spike trains from the raw HDF5.
    'minimal_output_downsample_saved_traces': True,  # If True, saved derived CST/sync traces are downsampled before writing the sidecar HDF5.
    'minimal_output_target_fsamp': 100,  # Target sampling rate, in Hz, for downsampled derived traces when minimal_output_downsample_saved_traces=True.
}

# BATCH-SUMMARY FIGURE CONFIGURATION
EXPORT_BATCH_SUMMARY_FIGURES = True  # If False, only the CSV/PKL exports are produced; no participant/session/condition summary PNGs are written.
SHOW_INDIVIDUAL_TRACES = False  # If True, draw each trial behind the batch mean in the summary figure.
SHOW_STD_BAND = True  # If True, draw mean ± SD shading around the batch-mean trace.
SHOW_REFERENCE_STD_LINES = True  # If True, draw horizontal ± mean-trial-SD reference lines on the CST panels.
FIX_YLIM_ACROSS_BATCHES = True  # If True, use common y-limits across participant/session/condition summary figures for each subplot type.
COLOR_OVERRIDES = {
    "lf": "#ff3300",
    "alpha": "#ffa600",
    "beta": "#008cff",
    "cst": "#000000",
    "sync": "#7e2179",
    "envelope": "#383838",
    "burst": "#E900CA",
    "cue": "#7a7a7a",
}
PLOT_STYLE = {
    'figure_size': (16, 11),
    'figure_dpi': 120,
    'feature_outside_color': '#C9C9C9',  # Background color used outside the feature-extraction window on analyzed panels.
    'feature_outside_alpha': 0.20,  # Background alpha used outside the feature-extraction window on analyzed panels.
    'feature_baseline_color': '#111111',  # Color of the fitted spline / baseline curve overlay used by the feature-extraction panels.
    'feature_baseline_lw': 1.4,  # Line width of the fitted spline / baseline curve overlay.
    'feature_baseline_ls': '--',  # Line style of the fitted spline / baseline curve overlay.
    'feature_analysis_mean_lw': 1.4,  # Line width of the horizontal line showing the mean of the displayed data inside the feature-analysis window.
    'feature_analysis_mean_ls': '--',  # Line style of the horizontal line showing the mean of the displayed data inside the feature-analysis window.
    'feature_analysis_mean_alpha': 1.0,  # Alpha of the horizontal line showing the mean of the displayed data inside the feature-analysis window.
}
RATE_ISI_DENSITY_CONFIG = {
    'enabled': True,  # If True, the top-left subplot shows pooled motor-unit firing-rate and ISI-CV densities across trials in the current batch.
    'hist_bins': 40,  # Number of bins used if use_kde=False.
    'use_kde': True,  # If True, draw a smoothed KDE density instead of a histogram-derived density curve.
    'kde_bw_method': None,  # Bandwidth selector passed to scipy.stats.gaussian_kde. None uses the default Scott rule.
    'normalize_each_density_to_peak': True,  # If True, each density curve is divided by its own peak so firing-rate and ISI-CV shapes use a comparable vertical scale despite different x axes.
    'line_lw': 1.8,  # Line width of the firing-rate and ISI-CV density curves.
    'firing_rate_color': "#FF7300",  # Color of the firing-rate density curve, fill, and summary text.
    'firing_rate_fill_alpha': 0.18,  # Fill alpha under the firing-rate density curve.
    'isi_cv_color': "#00CFAD",  # Color of the ISI-CV density curve, fill, and summary text.
    'isi_cv_fill_alpha': 0.18,  # Fill alpha under the ISI-CV density curve.
    'ylabel': 'Normalized density (peak = 1)',  # Shared y-axis label of the density panel.
    'bottom_xlabel': 'Mean firing rate (Hz)',  # Bottom x-axis label for the firing-rate density.
    'top_xlabel': 'ISI CV',  # Top x-axis label for the ISI-CV density.
    'title': 'MN firing-rate / ISI-CV distributions',  # Subplot title for the dual-density panel.
    'stats_fontsize': 8.5,  # Font size of the colored text showing mean / SD / skew averaged across trials.
}
WINDOW_OVERLAY_SPECS = {
    'firing_rate_isi_window_rel_cue_s': {'enabled': True, 'label': 'FR / ISI window', 'color': '#37a055', 'lw': 2.0, 'alpha': 1, 'y_axes': 0.02},
    'cst_modulation_normalization_window_rel_cue_s': {'enabled': True, 'label': 'CST normalization window', 'color': "#ffa600", 'lw': 2.0, 'alpha': 0.95, 'y_axes': 0.05},
    'sync_baseline_win_rel_cue_s': {'enabled': True, 'label': 'Sync baseline window', 'color': "#7e2179", 'lw': 2.0, 'alpha': 1, 'y_axes': 0.08},
    'feature_fitting_window': {'enabled': True, 'label': 'Spline fitting window', 'color': "#E09ED8", 'lw': 2, 'alpha': 1, 'y_axes': 0.15},
    'feature_analysis_window': {'enabled': True, 'label': 'Feature window', 'color': "#E900CA", 'lw': 2.0, 'alpha': 1, 'y_axes': 0.15},
}
MODULATION_SPECTRUM_CONFIG = {
    'enabled': True,  # If True, the first column shows the PSD of the z-scored alpha and beta Hilbert-envelope modulation over the dedicated spectrum window below.
    'window_rel_cue_s': (-3.0, -1.0),  # Cue-relative window used for the saved per-trial alpha/beta modulation spectra and their exported OBS_SPECTRUM_* features.
    'max_freq_hz': 30.0,  # LIMIT FOR ANALYSIS # Maximum frequency retained for both plotting and PSD-based summary metrics (peak frequency and smallest contiguous mass interval).
    'interval_mass_pct': 80.0,  # Percentage of total PSD mass captured by the smallest contiguous interval reported on the alpha / beta spectrum panels.
    'show_interval': True,  # VISUAL ONLY # If True, shade the selected contiguous PSD-mass interval and label its frequency bounds on the spectrum panels.
    'interval_alpha': 0.2,  # VISUAL ONLY # Alpha used for the shaded PSD-mass interval under the mean spectrum.
    'ylabel': 'PSD of baseline-zscored envelope (a.u.^2/Hz)',  # Shared y-axis label for the alpha / beta modulation spectrum panels.
}
ANALYSIS_CONFIG['modulation_spectrum_window_rel_cue_s'] = MODULATION_SPECTRUM_CONFIG['window_rel_cue_s']  # Sidecar-analysis window used when saving per-trial modulation-spectrum features.
ANALYSIS_CONFIG['modulation_spectrum_max_freq_hz'] = MODULATION_SPECTRUM_CONFIG['max_freq_hz']  # Sidecar-analysis frequency cutoff used when saving per-trial modulation-spectrum peak / interval features.
ANALYSIS_CONFIG['modulation_spectrum_interval_mass_pct'] = MODULATION_SPECTRUM_CONFIG['interval_mass_pct']  # Sidecar-analysis PSD-mass interval used when saving per-trial modulation-spectrum interval features.

# FEATURE EXTRACTION ON THE MEAN BATCH TRACES
FEATURE_EXTRACTION_CONFIG = {
    'enabled': True,  # Master switch for fitted-baseline residual feature extraction on the batch-mean CST LF, alpha modulation, beta modulation, and synchrony traces.
    'fitting_window': (-3.0, 1.0), # (-4.0, 1.0),  # Cue-relative interval over which the baseline spline is fit and evaluated.
    'analysis_window': (0.0, 0.5),  # Cue-relative interval in which the signed mean residual feature is measured after subtracting the fitted baseline.
    'buffer_s': 0.05,  # Samples inside analysis_window ± buffer_s are excluded from the spline fit, but the feature itself is still computed on analysis_window only.
    'spline_smoothing': None,  # Smoothing strength for UnivariateSpline. None uses the project default heuristic; larger values force a smoother, less flexible baseline.
    'spline_order': 4, # 3  # Spline degree. 3 means cubic spline, which is the default baseline model used in the feature-extraction overlay.
    'n_knots': None,  # If set, fit an LSQ spline with this many interior knots instead of the smoothing-spline path. Leave as None unless you explicitly want knot-controlled flexibility.
    'trend_pre_window': (-3.0, -1.0), # (-4.0, 0.0),  # Cue-relative pre-cue interval on the fitted spline used for the slow-trend feature: mean(fitted post window) - mean(fitted pre window).
    'trend_post_window': (0.0, 1.0),  # Cue-relative post-cue interval on the fitted spline used for the slow-trend feature. This affects OBS_TREND_FEATURE exports and the split fitted-line overlay.
    'extrapolate': False,  # If False, the spline is not allowed to extrapolate outside its support. This mainly matters if you later choose fitting/display windows near the data edges.
}

In [ ]:
analysis_runs_df = analyze_experimental_manifest(
    EXPERIMENTAL_INPUT_ROOT,
    EXPERIMENTAL_ANALYZED_ROOT,
    manifest_csv_path=MANIFEST_CSV_PATH,
    analysis_config=ANALYSIS_CONFIG,
    exported_only=True,
    participant_ids=PARTICIPANT_IDS,
    session_tags=SESSION_TAGS,
    condition_ids=CONDITION_IDS,
    max_trials=MAX_TRIALS,
    overwrite=False,
    show_progress=True,
)
analysis_runs_df.head()

In [ ]:
experimental_batch_summary_df = summarize_experimental_batches(
    EXPERIMENTAL_ANALYZED_ROOT,
    participant_ids=PARTICIPANT_IDS,
    session_tags=SESSION_TAGS,
    condition_ids=CONDITION_IDS,
    max_batches=MAX_BATCHES,
    show_individual_traces=SHOW_INDIVIDUAL_TRACES,
    show_std_band=SHOW_STD_BAND,
    show_reference_std_lines=SHOW_REFERENCE_STD_LINES,
    fixed_ylim_across_batches=FIX_YLIM_ACROSS_BATCHES,
    color_overrides=COLOR_OVERRIDES,
    plot_style=PLOT_STYLE,
    xlim_rel_cue_s=DISPLAY_XLIM_REL_CUE_S,
    feature_config=FEATURE_EXTRACTION_CONFIG,
    modulation_spectrum_config=MODULATION_SPECTRUM_CONFIG,
    rate_isi_density_config=RATE_ISI_DENSITY_CONFIG,
    window_overlay_specs=WINDOW_OVERLAY_SPECS,
    export_figures=(EXPORT_BATCH_SUMMARY_FIGURES and not (ANALYSIS_CONFIG['minimal_output'] and ANALYSIS_CONFIG['minimal_output_turn_off_figure_output'])),
    show_progress=True,
)

experimental_rate_isi_metrics_df, experimental_rate_isi_skipped = collect_experimental_rate_isi_moments(
    EXPERIMENTAL_ANALYZED_ROOT,
    participant_ids=PARTICIPANT_IDS,
    session_tags=SESSION_TAGS,
    condition_ids=CONDITION_IDS,
    max_batches=MAX_BATCHES,
    show_progress=True,
)

experimental_batch_summary_df.head()

In [ ]:
experimental_export_df, experimental_export_paths = export_experimental_summary_table(
    EXPERIMENTAL_ANALYZED_ROOT,
    batch_summary_df=experimental_batch_summary_df,
    rate_isi_metrics_df=experimental_rate_isi_metrics_df,
    feature_config=FEATURE_EXTRACTION_CONFIG,
    modulation_spectrum_config=MODULATION_SPECTRUM_CONFIG,
    participant_ids=PARTICIPANT_IDS,
    session_tags=SESSION_TAGS,
    condition_ids=CONDITION_IDS,
    max_batches=MAX_BATCHES,
)

print(experimental_export_paths)
experimental_export_df.head()